In [2]:
import pandas as pd

# ==========================================
# 1. 데이터 불러오기 및 전처리
# ==========================================
base_path = "/Users/wonwoo_mac/Desktop/SPARTA_PROJECT_4TH_E_COMMERCE/송원우/data/"
womens_best_file = base_path + "lululemon_womens_bestseller_20260416.csv"
womens_review_file = base_path + "lululemon_womens_bestseller_20260416_reviews.csv"

# 데이터 로드
df_w_best = pd.read_csv(womens_best_file)
df_w_review = pd.read_csv(womens_review_file)

# 여성용 데이터 'price' -> 'min_price' 로 이름 변경 및 N/A 제거
df_w_best = df_w_best.rename(columns={'price': 'min_price'})
df_w_best = df_w_best[df_w_best['product_id'] != 'N/A'].dropna(subset=['product_id'])

print(f"📦 여성 상품 데이터: {len(df_w_best)}개 / 리뷰 데이터: {len(df_w_review)}개 로드 완료")


# ==========================================
# 2. 데이터 병합 (Merge)
# ==========================================
df_w_merged = pd.merge(
    df_w_review, 
    df_w_best[['product_id', 'product_name', 'category', 'min_price']], 
    on='product_id', 
    how='left'
)

# 상품 정보가 매칭되지 않는 껍데기 리뷰 제외
df_w_merged = df_w_merged.dropna(subset=['product_name'])

print(f"🔗 데이터 병합 완료! 유효한 분석 대상 리뷰: {len(df_w_merged)}개\n")


# ==========================================
# 3. 상세 통계 분석 (EDA)
# ==========================================
print("📊 [여성 베스트셀러 통계 요약]")
print("-" * 50)

# ① 전체 평균 평점
avg_rating = df_w_merged['rating'].mean()
print(f"⭐ 전체 평균 평점: {avg_rating:.2f} / 5.0")

# ② 카테고리별 성과 (에러 해결 부분 💡)
# 'review_id' 대신 무조건 존재하는 'rating' 컬럼의 개수를 세도록 변경했습니다.
category_stats = df_w_merged.groupby('category').agg(
    review_count=('rating', 'count'), # 👈 여기를 수정했습니다!
    avg_rating=('rating', 'mean')
).sort_values(by='review_count', ascending=False)

print("\n📈 [카테고리별 성과 TOP 10]")
display(category_stats.head(10))

# ③ 불만족 리뷰 분석 (1~3점)
low_rating_df = df_w_merged[df_w_merged['rating'] <= 3]
low_ratio = (len(low_rating_df) / len(df_w_merged)) * 100
print(f"\n⚠️ 불만족 리뷰(3점 이하) 비중: {low_ratio:.1f}% ({len(low_rating_df)}건)")

# ④ 가장 인기 있는(리뷰가 많은) 개별 상품 TOP 5
top_5_products = df_w_merged['product_name'].value_counts().head(5)
print("\n🔥 [가장 많은 리뷰가 달린 인기 상품 TOP 5]")
print(top_5_products)


# ==========================================
# 4. 결과 저장
# ==========================================
merged_output = base_path + "lululemon_womens_merged_analysis.csv"
df_w_merged.to_csv(merged_output, index=False, encoding='utf-8-sig')
print(f"\n💾 병합된 데이터가 저장되었습니다: {merged_output}")

📦 여성 상품 데이터: 93개 / 리뷰 데이터: 18902개 로드 완료
🔗 데이터 병합 완료! 유효한 분석 대상 리뷰: 18902개

📊 [여성 베스트셀러 통계 요약]
--------------------------------------------------
⭐ 전체 평균 평점: 4.66 / 5.0

📈 [카테고리별 성과 TOP 10]


,review_count,avg_rating
category,,
womens-leggings,2747,4.566800
womens-shorts,2404,4.690100
womens-hoodies-and-sweatshirts,2136,4.694288
womens-tank-tops,1870,4.617647
womens-jackets-and-outerwear,1697,4.720094
womens-casual-pants,1592,4.655151
womens-short-sleeve-tops,1253,4.749401
bags,1234,4.834684
womens-sports-bras,813,4.324723



⚠️ 불만족 리뷰(3점 이하) 비중: 8.2% (1558건)

🔥 [가장 많은 리뷰가 달린 인기 상품 TOP 5]
product_name
룰루레몬 Align™ 하이라이즈 팬츠 25"          1407
룰루레몬 Align™ 하이라이즈 쇼츠 6"           1255
스쿠바 오버사이즈드 풀집                      860
그루브 Nulu 슈퍼 하이라이즈 플레어드 팬츠 *레귤러     846
에브 투 스트리트 탱크탑                      725
Name: count, dtype: int64

💾 병합된 데이터가 저장되었습니다: /Users/wonwoo_mac/Desktop/SPARTA_PROJECT_4TH_E_COMMERCE/송원우/data/lululemon_womens_merged_analysis.csv


In [3]:
import pandas as pd

# 📂 경로 설정
base_path = "/Users/wonwoo_mac/Desktop/SPARTA_PROJECT_4TH_E_COMMERCE/송원우/data/"

# ==========================================
# 1. 남성 데이터 처리 (Merge + Gender 추가)
# ==========================================
df_m_best = pd.read_csv(base_path + "lululemon_mens_bestseller_20260416.csv")
df_m_review = pd.read_csv(base_path + "lululemon_mens_bestseller_reviews.csv")

# 남성 데이터 병합
df_m_merged = pd.merge(
    df_m_review, 
    df_m_best[['product_id', 'product_name', 'category', 'min_price']], 
    on='product_id', how='left'
).dropna(subset=['product_name'])

# 💡 성별 컬럼 추가 (Men: 0)
df_m_merged['gender'] = 0 
df_m_merged['gender_label'] = 'Men'


# ==========================================
# 2. 여성 데이터 처리 (Merge + Gender 추가)
# ==========================================
df_w_best = pd.read_csv(base_path + "lululemon_womens_bestseller_20260416.csv")
df_w_review = pd.read_csv(base_path + "lululemon_womens_bestseller_20260416_reviews.csv")

# 여성용 데이터 'price' -> 'min_price' 로 통일
df_w_best = df_w_best.rename(columns={'price': 'min_price'})

# 여성 데이터 병합
df_w_merged = pd.merge(
    df_w_review, 
    df_w_best[['product_id', 'product_name', 'category', 'min_price']], 
    on='product_id', how='left'
).dropna(subset=['product_name'])

# 💡 성별 컬럼 추가 (Women: 1)
df_w_merged['gender'] = 1
df_w_merged['gender_label'] = 'Women'


# ==========================================
# 3. 통합 파일 생성 (Concatenate)
# ==========================================
# 두 데이터프레임을 위아래로 이어 붙입니다.
df_total = pd.concat([df_m_merged, df_w_merged], ignore_index=True)

print(f"✅ 통합 완료! 전체 데이터 수: {len(df_total)}행")
print(f"👨 남성 리뷰: {len(df_m_merged)}개 / 👩 여성 리뷰: {len(df_w_merged)}개")


# ==========================================
# 4. 성별 비교 통계 (EDA)
# ==========================================
print("\n📊 [성별 만족도 및 가격 비교]")
print("-" * 50)

# 성별에 따른 평균 평점과 평균 가격 비교
gender_comparison = df_total.groupby('gender_label').agg(
    avg_rating=('rating', 'mean'),
    avg_price=('min_price', 'mean'),
    review_count=('rating', 'count')
)
display(gender_comparison)

# 최종 통합 데이터 저장
total_output = base_path + "lululemon_integrated_analysis.csv"
df_total.to_csv(total_output, index=False, encoding='utf-8-sig')
print(f"\n💾 통합 분석용 파일이 저장되었습니다: {total_output}")

✅ 통합 완료! 전체 데이터 수: 23076행
👨 남성 리뷰: 4174개 / 👩 여성 리뷰: 18902개

📊 [성별 만족도 및 가격 비교]
--------------------------------------------------


,avg_rating,avg_price,review_count
gender_label,,,
Men,4.703162,97019.166267,4174
Women,4.655116,108011.268649,18902



💾 통합 분석용 파일이 저장되었습니다: /Users/wonwoo_mac/Desktop/SPARTA_PROJECT_4TH_E_COMMERCE/송원우/data/lululemon_integrated_analysis.csv


In [6]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 🍎 맥북 환경 한글 폰트 및 마이너스 기호 깨짐 방지 설정
plt.rc('font', family='AppleGothic')
plt.rcParams['axes.unicode_minus'] = False

# ==========================================
# 1. 데이터 로드
# ==========================================
base_path = "/Users/wonwoo_mac/Desktop/SPARTA_PROJECT_4TH_E_COMMERCE/송원우/data/"
filename = "lululemon_bestseller_merged.csv"

df = pd.read_csv(base_path + filename)
print(f"📦 통합 데이터 로드 완료: 총 {len(df)}건")


# ==========================================
# 2. 카테고리별 고객 참여도 (리뷰 수) 분석
# ==========================================
plt.figure(figsize=(12, 6))
# 상위 10개 카테고리만 추출
top_categories = df['category'].value_counts().nlargest(10).index
sns.countplot(
    data=df[df['category'].isin(top_categories)], 
    y='category', 
    hue='gender_label', 
    order=top_categories, 
    palette='muted'
)
plt.title('🏆 룰루레몬 상위 10개 카테고리별 리뷰 수 (성별 비교)', fontsize=16, fontweight='bold', pad=15)
plt.xlabel('리뷰 수 (고객 참여도)', fontsize=12)
plt.ylabel('카테고리', fontsize=12)
plt.legend(title='성별')
plt.tight_layout()
plt.show()


# ==========================================
# 3. 성별에 따른 가격대 분포 (Boxplot)
# ==========================================
plt.figure(figsize=(8, 6))
sns.boxplot(data=df, x='gender_label', y='min_price', width=0.5, palette='pastel')
plt.title('💰 성별 베스트셀러 가격대 분포', fontsize=16, fontweight='bold', pad=15)
plt.xlabel('성별', fontsize=12)
plt.ylabel('최저가 (KRW)', fontsize=12)

# Y축 가격 단위를 보기 좋게 (천원 단위) 변경
current_values = plt.gca().get_yticks()
plt.gca().set_yticklabels(['{:,.0f}원'.format(x) for x in current_values])

plt.tight_layout()
plt.show()


# ==========================================
# 4. 가격과 평점의 상관관계 (포지셔닝 맵)
# ==========================================
# 상품별로 평균 평점, 최저가, 리뷰 수를 집계합니다.
product_stats = df.groupby(['product_name', 'gender_label']).agg(
    avg_rating=('rating', 'mean'),
    price=('min_price', 'min'),
    review_count=('rating', 'count')
).reset_index()

# 리뷰가 너무 적은 상품은 노이즈가 될 수 있으므로 제외 (예: 리뷰 3개 이상만)
product_stats = product_stats[product_stats['review_count'] >= 3]

plt.figure(figsize=(10, 7))
# 리뷰 수(review_count)를 버블의 크기(size)로 설정
sns.scatterplot(
    data=product_stats, 
    x='price', 
    y='avg_rating', 
    hue='gender_label',
    size='review_count', 
    sizes=(50, 1000), # 버블 크기 최소/최대
    alpha=0.6,
    palette='Set2'
)
plt.title('🎯 가격 vs 만족도 포지셔닝 맵 (버블 크기: 리뷰 수)', fontsize=16, fontweight='bold', pad=15)
plt.xlabel('가격 (KRW)', fontsize=12)
plt.ylabel('평균 평점 (1~5점)', fontsize=12)

# Y축 한계 설정 (평점이 주로 3~5점 사이에 몰려있으므로)
plt.ylim(3.0, 5.2)

# X축 가격 포맷팅
current_values = plt.gca().get_xticks()
plt.gca().set_xticklabels(['{:,.0f}원'.format(x) for x in current_values])

# 범례가 그래프를 가리지 않도록 밖으로 이동
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

📦 통합 데이터 로드 완료: 총 23076건


ModuleNotFoundError: No module named 'IPython.core.pylabtools'